Step 1: Loading and Initial Review (Preview)
Goal: Check the file structure, column names, and first few rows without overloading RAM (memory).

Крок 1: Завантаження та первинний огляд (Preview)
Мета: Перевірити структуру файлу, назви колонок і перші рядки, не перевантажуючи оперативну пам'ять.

In [4]:
import pandas as pd

# Define the correct dataset file path
# Вказуємо точний шлях до CSV-файлу
file_path = "data/PS_20174392719_1491204439457_log.csv"

# Load the first 5 rows to inspect column names and structure
# Завантажуємо перші 5 рядків для перевірки структури та назв колонок
df_preview = pd.read_csv(file_path, nrows=5)

# Display the preview
# Виводимо перші 5 рядків
print("--- Dataset Preview (First 5 Rows) / Перші 5 рядків датасету ---")
display(df_preview)

--- Dataset Preview (First 5 Rows) / Перші 5 рядків датасету ---


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


Step 2: Checking Data Size and Types (Metadata)
Goal: Determine the total number of rows, the data type of each column, and the presence of missing values (Null/NaN).

Крок 2: Перевірка розміру та типів даних (Metadata)
Мета: Дізнатися загальну кількість рядків, тип кожної колонки та наявність пропущених значень (Null/NaN).

In [5]:
# Load the full dataset into memory
# Завантажуємо повний датасет в оперативну пам'ять
df = pd.read_csv(file_path)

# Display total rows and columns
# Виводимо загальну кількість рядків і колонок
print("--- Dataset Shape / Розмір датасету ---")
print(f"Total rows (Всього рядків): {df.shape[0]:,}")
print(f"Total columns (Всього колонок): {df.shape[1]}")

# Inspect column data types and missing values
# Перевіряємо типи даних колонок та наявність пропусків
print("\n--- Data Types & Missing Values / Типи даних та пропуски ---")
print(df.info())

--- Dataset Shape / Розмір датасету ---
Total rows (Всього рядків): 6,362,620
Total columns (Всього колонок): 11

--- Data Types & Missing Values / Типи даних та пропуски ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB
None


Step 3: Target Variable and Fraud Analysis (Target Distribution)
Goal: See how many total fraudulent transactions are in the dataset and what percentage they represent (class imbalance).

Крок 3: Аналіз цільової змінної та шахрайства (Target Distribution)
Мета: Подивитися, скільки всього шахрайських транзакцій у датасеті та який відсоток вони складають (дисбаланс класів).

In [6]:
# Count legit (0) vs fraudulent (1) transactions
# Підраховуємо кількість легітимних (0) та шахрайських (1) транзакцій
fraud_counts = df['isFraud'].value_counts()

# Calculate percentages
# Розраховуємо відсоткове співвідношення
fraud_percentage = df['isFraud'].value_counts(normalize=True) * 100

print("--- Fraud vs Legit Transactions / Шахрайські та легітимні транзакції ---")
print("Counts (Кількість):")
print(fraud_counts)
print("\nPercentage (Відсотки):")
print(fraud_percentage.round(4).astype(str) + " %")

--- Fraud vs Legit Transactions / Шахрайські та легітимні транзакції ---
Counts (Кількість):
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Percentage (Відсотки):
isFraud
0    99.8709 %
1     0.1291 %
Name: proportion, dtype: object


Step 4: Fraud Analysis by Transaction Types (Transaction Types)
Goal: Identify which specific types of operations (TRANSFER, CASH_OUT, PAYMENT, etc.) contain fraud.

Крок 4: Аналіз шахрайства за типами транзакцій (Transaction Types)
Мета: З'ясувати, у яких саме типах операцій (TRANSFER, CASH_OUT, PAYMENT тощо) виникає шахрайство.

In [7]:
# Group by transaction type and calculate total count and fraud count
# Групуємо за типом транзакції та обчислюємо загальну кількість і кількість шахрайства
type_analysis = df.groupby('type').agg(
    total_transactions=('isFraud', 'count'),
    fraud_transactions=('isFraud', 'sum')
).reset_index()

# Calculate fraud rate per type (%)
# Обчислюємо відсоток шахрайства для кожного типу (%)
type_analysis['fraud_rate_%'] = (type_analysis['fraud_transactions'] / type_analysis['total_transactions'] * 100).round(4)

print("--- Fraud Analysis by Transaction Type / Аналіз шахрайства за типами транзакцій ---")
display(type_analysis)

--- Fraud Analysis by Transaction Type / Аналіз шахрайства за типами транзакцій ---


,type,total_transactions,fraud_transactions,fraud_rate_%
0,CASH_IN,1399284,0,0.0000
1,CASH_OUT,2237500,4116,0.1840
2,DEBIT,41432,0,0.0000
3,PAYMENT,2151495,0,0.0000
4,TRANSFER,532909,4097,0.7688


### Step 5. Feature Engineering (Створення похибок балансу)

## Відфільтровуємо лише ті типи транзакцій, де реально є шахрайство (TRANSFER та CASH_OUT), відкидаючи зайвий шум.
## Рахуємо errorBalanceOrig та errorBalanceDest — це ключові індикатори для антифроду (шахраї часто обнуляють рахунки не так, як це відбувається при легальних операціях).

In [8]:
# 1. Filter only TRANSFER and CASH_OUT transactions where fraud actually occurs
# Фільтруємо лише типи TRANSFER та CASH_OUT, де трапляється шахрайство
df_filtered = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()

# 2. Feature Engineering: Calculate balance errors
# Створюємо нові фічі: похибки зміни балансу відправника та отримувача
df_filtered["errorBalanceOrig"] = (
    df_filtered["oldbalanceOrg"]
    - df_filtered["amount"]
    - df_filtered["newbalanceOrig"]
)
df_filtered["errorBalanceDest"] = (
    df_filtered["oldbalanceDest"]
    + df_filtered["amount"]
    - df_filtered["newbalanceDest"]
)

print(
    "--- Feature Engineering Completed / Інжиніринг ознак завершено ---"
)
print(f"Filtered dataset shape (Розмір після фільтрації): {df_filtered.shape}")
display(df_filtered[["type", "amount", "errorBalanceOrig", "errorBalanceDest", "isFraud"]].head(3))

--- Feature Engineering Completed / Інжиніринг ознак завершено ---
Filtered dataset shape (Розмір після фільтрації): (2770409, 13)


,type,amount,errorBalanceOrig,errorBalanceDest,isFraud
2,TRANSFER,181.00,0.00,181.0,1
3,CASH_OUT,181.00,0.00,21363.0,1
15,CASH_OUT,229133.94,-213808.94,182703.5,0


### Step 6: Data Encoding and Train/Test Split
## Goal: Prepare the dataset for Machine Learning by converting categorical variables into numbers, dropping identifier columns, and splitting data into Training (80%) and Testing (20%) sets.

## Мета: Підготувати дані для ML: перетворити текстові змінні у числові, вилучити непотрібні ідентифікатори та розділити вибірку на Навчальну (80%) і Тестову (20%).

In [9]:
from sklearn.model_selection import train_test_split

# 1. One-hot encoding for transaction type (TRANSFER vs CASH_OUT)
# Перетворюємо текстовий тип у числовий
df_ml = pd.get_dummies(df_filtered, columns=['type'], drop_first=True, dtype=int)

# 2. Select Features (X) and Target (y)
# Видаляємо текстові ідентифікатори, які не використовуються моделю
drop_cols = ['isFraud', 'nameOrig', 'nameDest', 'isFlaggedFraud']
X = df_ml.drop(columns=drop_cols)
y = df_ml['isFraud']

# 3. Perform Stratified Train-Test Split (80% Train, 20% Test)
# Розділяємо дані з збереженням пропорції шахрайських транзакцій
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("--- Step 6 Completed: Train/Test Split / Розподіл завершено ---")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Fraud count in Train: {y_train.sum()} ({y_train.mean()*100:.3f}%)")
print(f"Fraud count in Test: {y_test.sum()} ({y_test.mean()*100:.3f}%)")

--- Step 6 Completed: Train/Test Split / Розподіл завершено ---
X_train shape: (2216327, 9)
X_test shape: (554082, 9)
Fraud count in Train: 6570 (0.296%)
Fraud count in Test: 1643 (0.297%)


### Step 7: Train XGBoost Classifier & Model Evaluation
## Goal: Train an XGBoost model optimized for extreme class imbalance using scale_pos_weight and evaluate its performance using Precision, Recall, F1-Score, and ROC-AUC metrics.

## Мета: Навчити модель XGBoost з урахуванням дисбалансу класів за допомогою ваги scale_pos_weight та оцінити якість роботи моделі за допомогою метрик Precision, Recall, F1-Score та ROC-AUC.

In [11]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.8 MB/s  0:00:00 eta 0:00:01


In [12]:
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# 1. Calculate class ratio for imbalanced weighting
# Розраховуємо коефіцієнт ваги для шахрайського класу
scale_weight = (len(y_train) - sum(y_train)) / sum(y_train)

# 2. Initialize and Train XGBoost Classifier
# Ініціалізуємо та навчаємо модель XGBoost
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1
)

print("--- Training XGBoost Model... / Навчання моделі XGBoost... ---")
model.fit(X_train, y_train)

# 3. Generate Predictions on Test Set
# Отримуємо прогнози для тестової вибірки
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# 4. Display Evaluation Results
# Виводимо результати оцінки моделі
print("\n--- Model Evaluation Report / Звіт про якість моделі ---")
print(classification_report(y_test, y_pred, digits=4))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

--- Training XGBoost Model... / Навчання моделі XGBoost... ---

--- Model Evaluation Report / Звіт про якість моделі ---
              precision    recall  f1-score   support

           0     1.0000    0.9999    0.9999    552439
           1     0.9562    0.9976    0.9765      1643

    accuracy                         0.9999    554082
   macro avg     0.9781    0.9987    0.9882    554082
weighted avg     0.9999    0.9999    0.9999    554082

ROC-AUC Score: 0.9991
